# ラボ 1a: 出生（natality）データセットの探索

**学習目標**

1. BigQuery を使用して出生（natality）データセットを探索する
1. Vertex AI Workbench（旧 Cloud AI Platform）ノートブックを使用してデータの探索結果をプロットする


## はじめに 
このノートブックでは、生まれる前の赤ちゃんの体重を予測するモデルの開発とトレーニングを開始する前に、出生（natality）データセットを探索します。BigQuery を使用してデータを探索し、Vertex AI ノートブックを使用してデータ探索結果をプロットします。

各学習目標は、この受講者用ラボ ノートブック内の __#TODO__ に対応しています。まずこのノートブックを完了してみてから、[解答ノートブック](../solutions/1a_explore_data_babyweight.ipynb) を確認してください。

## 必要なライブラリの読み込み

In [ ]:
from google.cloud import bigquery

## ソース データセット

データセットは [BigQuery](https://cloud.google.com/bigquery/) でホストされています。CDC の Natality データには 1969 年から 2008 年までの米国での出生に関する詳細が含まれており、公開データセットであるため、Google Cloud アカウントを持つ誰でもアクセスできます。データセットにアクセスするには、[こちら](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) をクリックしてください。

出生データセットは約 1 億 3,800 万行、31 列と比較的規模が大きいですが、理解しやすい構成になっています。`weight_pounds` はターゲットであり、モデルをトレーニングして予測する連続値です。

<h2> データの探索 </h2>

このデータは出生データ（米国における出生記録）です。目的は、妊娠と赤ちゃんの母親に関する多くの要因を前提として、赤ちゃんの体重を予測することです。後でデータをトレーニング用と評価用のデータセットに分割します。その際には年月のハッシュが使用されます。これにより、同じ日に生まれた双子がデータの異なる分割に分かれるのを防ぎます。まず、2000 年以降の出生データを使用して SQL クエリを作成します。

In [ ]:
query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    FARM_FINGERPRINT(
        CONCAT(
            CAST(YEAR AS STRING),
            CAST(month AS STRING)
        )
    ) AS hashmonth
FROM
    publicdata.samples.natality
WHERE
    year > 2000
"""

ノートブック全体で使用できる BigQuery クライアントを作成しましょう。

In [ ]:
bq = bigquery.Client()

新しく作成したクライアントを使用して、BigQuery の呼び出し結果を Pandas DataFrame で確認してみましょう。

In [ ]:
# BigQuery を呼び出し、データフレームで確認
df = bigquery.Client().query(query + " LIMIT 100").to_dataframe()
df.head()

まず、出生データセット内のすべての有効な列名のセットを取得しましょう。これを行うには、データセットのテーブルの `INFORMATION_SCHEMA` にアクセスします。

In [ ]:
# テーブル スキーマ内のすべての列名を取得するクエリ
sql = """
SELECT
    column_name
FROM
    publicdata.samples.INFORMATION_SCHEMA.COLUMNS
WHERE
    table_name = "natality"
"""

# BigQuery クライアント経由でクエリを送信し、出力をデータフレームに保存
valid_columns_df = bq.query(sql).to_dataframe()

# データフレーム内の列名をセットに変換
valid_columns_set = valid_columns_df["column_name"].tolist()

有効な列セットを出力して、データセットで利用可能なすべての列を確認できます。もちろん、[BigQuery UI](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) でテーブルを選択したときに [Schema]（スキーマ）タブに移動してこの情報を確認することもできます。

In [ ]:
print(valid_columns_set)

## ラボ課題 1: BigQuery を使用した出生データセットの探索
上記のコードを例として、2000 年以降に生まれた赤ちゃんについて、各列の一意の値とその値のカウントを求めるクエリを作成してください。
たとえば、次のような値を取得します。
<pre>
is_male	num_babies	avg_wt
False	16245054	7.104715
 True	17026860	7.349797
</pre>
これは、各データ値の十分なサンプルがあることを確認し、そのパラメータに予測価値があるという仮説を検証するために重要です。

ヒント（強調表示して表示）: <p style='color:white'>COUNT()、AVG()、GROUP BY を使用します。例:
<pre style='color:white'>
SELECT
  is_male,
  COUNT(1) AS num_babies,
  AVG(weight_pounds) AS avg_wt
FROM
  publicdata.samples.natality
WHERE
  year > 2000
GROUP BY
  is_male
</pre>
</p>

In [ ]:
# TODO: BigQuery から個別値の統計情報を取得する関数を作成する
def get_distinct_values(valid_columns_set, column_name):
    """BigQuery データ列の個別値の統計情報を取得します。

    Args:
        valid_columns_set: set、テーブル内のすべての可能な有効な列名のセット。
        column_name: str、BigQuery 内の列名。
    Returns:
        一意の値、そのカウント、および平均のデータフレーム。
    """
    assert (
        column_name in valid_columns_set
    ), "{column_name} is not a valid column_name".format(
        column_name=column_name
    )

    sql = """
    """

    pass

## ラボ課題 2: ノートブックを使用した探索結果のプロット

どの要因が赤ちゃんの体重に影響を与えているように見えますか？

<b>ボーナス:</b> 結論を説明するためのグラフを描画してください

ヒント（強調表示して表示）: 
<p style='color:white'># TODO: 実装したばかりの get_distinct_values 関数を再利用して、BigQuery からの個別値の統計情報をプロットする関数を作成します

ヒント（強調表示して表示）: <p style='color:white'> プロットする最も簡単な方法は、Pandas の組み込みプロット機能を使用することです
<pre style='color:white'>
df = get_distinct_values(valid_columns_set, column_name)
df = df.sort_values(column_name)
df.plot(x=column_name, y="num_babies", kind="bar", figsize=(12, 5))
df.plot(x=column_name, y="avg_wt", kind="bar", figsize=(12, 5))
</pre>

In [ ]:
# TODO: BigQuery からの個別値の統計情報をプロットする関数を作成する
def plot_distinct_values(valid_columns_set, column_name, logy=False):
    """BigQuery データ列の個別値の統計情報をプロットします。

    Args:
        valid_columns_set: set、テーブル内のすべての可能な有効な列名のセット。
        column_name: str、BigQuery 内の列名。
        logy: bool、カウントを対数スケールでプロットするかどうか。
    """
    pass

`is_male` について、`avg_wt` を線形スケールで、`num_babies` を対数スケールで表示する棒グラフを作成します。

In [ ]:
# TODO: Plot is_male

`mother_age` について、`avg_wt` を線形スケールで、`num_babies` を線形スケールで表示する棒グラフを作成します。

In [ ]:
# TODO: Plot mother_age

`plurality`（胎数）について、`avg_wt` を線形スケールで、`num_babies` を対数スケールで表示する棒グラフを作成します。

In [ ]:
# TODO: Plot plurality

`gestation_weeks`（妊娠週数）について、`avg_wt` を線形スケールで、`num_babies` を対数スケールで表示する棒グラフを作成します。

In [ ]:
# TODO: Plot gestation_weeks

これらすべての要因が赤ちゃんの体重に影響を与えているようです。男の子の赤ちゃんは女の子の赤ちゃんよりも平均して重いです。十代や高齢の母親は、体重の軽い赤ちゃんを産む傾向があります。双子や三つ子などは、単胎児よりも体重が軽くなります。未熟児は体重が軽く、シングルマザーから生まれた赤ちゃんも同様です。さらに、各入力値に対して十分なデータ（赤ちゃんの数）があるかどうかを確認することが重要です。そうでない場合、データが不十分な入力値に対するモデルの予測は信頼できない可能性があります。
<p>
次のノートブックでは、これらすべての要因を組み合わせて赤ちゃんの体重の予測を導き出す機械学習モデルを開発します。

## ラボのまとめ: 
このラボでは、BigQuery を使用してデータを探索し、Vertex AI ノートブックを使用してデータ探索結果をプロットしました。

Copyright 2020 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.